In [1]:
import os
import glob
import numpy as np
import open3d as o3d
from tqdm import tqdm
import time

DATASET_PATH = r"Novi Dataset\npy"    # tvojih 657 oblaka (58 stabala)
TRAVA = 3                              # oznaka klase trave (0=deblo,1=grane,2=potpora,3=trava)

# --- RANSAC parametri (identicni profesoricinom kodu) ---
distance_threshold = 0.06       # RANSAC prag udaljenosti za ravninu
ransac_n = 3                    # min tocaka za ravninu
num_iterations = 1000           # broj RANSAC iteracija
min_inlier_points = 30          # min inliera da se ravnina prihvati
angle_threshold = np.radians(20)  # +-20 stupnjeva (normala vs os visine)
plane_size = 3.0                # velicina ravnine (nije nuzno za mjerenje)
grass_treshold_distance = 0.35  # tocke blize ravnini od ovoga = trava
Z_tree_length_treshold = 1.5    # ako je raspon dubine > ovoga -> ima trave
BOTTOM_1 = 0.6                   # prvi filter donjeg dijela


print(f"RANSAC uklanjanje trave | {DATASET_PATH}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
RANSAC uklanjanje trave | Novi Dataset\npy


In [2]:
def ransac_ukloni_travu(points):
    """
    Doslovno prati profesoricin algoritam uklanjanja trave (Y=visina, Z=dubina).
    Umjesto rezanja vraca segmentaciju: 1 = trava, 0 = ne-trava.
    Mapiranje inliera radi se identicno njenom kodu (original_points_list.index).
    """
    N = points.shape[0]
    pred = np.zeros(N, dtype=np.int64)
    y_axis = np.array([0, 1, 0])   # vertikala = Y (kao kod nje)

    # --- uvjet postoji li trava: raspon po DUBINI (Z) ---
    abs_distance = abs(np.min(points[:,2]) - np.max(points[:,2]))
    if abs_distance <= Z_tree_length_treshold:
        return pred   # nema trave

    # --- radi samo na donjem dijelu (Y < y_min + 0.6) ---
    all_points = points
    mask = points[:,1] < (np.min(points[:,1]) + BOTTOM_1)
    filtered_points = points[mask]
    if len(filtered_points) < min_inlier_points:
        return pred

    remaining_pcd = o3d.geometry.PointCloud()
    remaining_pcd.points = o3d.utility.Vector3dVector(filtered_points)

    # njeno mapiranje: lista tuple-ova originalnih tocaka (donjeg dijela)
    original_points_list = [tuple(point) for point in np.asarray(remaining_pcd.points)]

    segmented_planes = []
    plane_meshes = []
    grass_cloud_indexes_list = []     # indeksi u prostoru filtered_points
    normal_vector_list = []
    counterForGrassPlanes = 0

    # --- njena while True petlja ---
    while True:
        try:
            plane_model, inliers = remaining_pcd.segment_plane(
                distance_threshold=distance_threshold,
                ransac_n=ransac_n,
                num_iterations=num_iterations
            )
        except Exception:
            inliers = list(range(len(remaining_pcd.points)))

        if len(inliers) < min_inlier_points:
            break

        normal_vector = np.array(plane_model[:3])
        normal_vector /= np.linalg.norm(normal_vector)
        angle_to_y_axis = np.arccos(np.dot(normal_vector, y_axis))

        inlier_cloud = remaining_pcd.select_by_index(inliers)
        centroid = np.mean(np.asarray(inlier_cloud.points), axis=0)

        if angle_to_y_axis <= angle_threshold:
            # njeno sporo mapiranje inliera na originalne indekse
            original_points_set = set(original_points_list)
            inlier_points = np.asarray(inlier_cloud.points)
            corresponding_original_indexes = [
                original_points_list.index(tuple(point))
                for point in inlier_points if tuple(point) in original_points_set
            ]
            grass_cloud_indexes_list.append(corresponding_original_indexes)
            normal_vector_list.append([normal_vector[0], normal_vector[1],
                                       normal_vector[2], -np.dot(normal_vector, centroid)])
            counterForGrassPlanes += 1

        # makni inliere
        remaining_pcd = remaining_pcd.select_by_index(inliers, invert=True)

    # --- ako je nadjena ravnina trave ---
    if grass_cloud_indexes_list:
        # ravnina s najvise tocaka (kao kod nje)
        maxGrassPoints = 0
        index_plane = 0
        for i in range(0, len(grass_cloud_indexes_list)):
            if maxGrassPoints <= len(grass_cloud_indexes_list[i]):
                maxGrassPoints = len(grass_cloud_indexes_list[i])
                index_plane = i

        # njeno prosirenje: udaljenost filtered_points od ravnine < prag = trava
        a, b, c = normal_vector_list[index_plane][:3]
        d = normal_vector_list[index_plane][3]
        distances_tree_from_plane = np.abs(
            a*filtered_points[:,0] + b*filtered_points[:,1] + c*filtered_points[:,2] + d
        ) / np.sqrt(a**2 + b**2 + c**2)
        partOfGrass = np.where(distances_tree_from_plane < grass_treshold_distance)[0]

        # mapiraj natrag na globalne indekse
        grass_indices_in_original = np.where(mask)[0][partOfGrass]
        pred[grass_indices_in_original] = 1

    else:
        # --- njen FALLBACK: nema ravnine, brisi po dubini (Z) ---
        points_ = np.asarray(all_points)
        mask_b = points_[:,1] < (np.min(points_[:,1]) + BOTTOM_1)
        mask_tree = points_[:,1] > (np.min(points_[:,1]) + BOTTOM_1)
        if mask_tree.sum() > 0:
            min_Z_fromTreeBranch = np.min(points_[mask_tree, 2])
            grass_points = np.where(points_[mask_b, 2] < min_Z_fromTreeBranch)[0]
            grass_indices_in_original = np.where(mask_b)[0][grass_points]
            pred[grass_indices_in_original] = 1

    return pred

In [3]:
def izracunaj_metrike(conf):
    """conf = 2x2 matrica zabune [stvarno][predvidjeno], klasa 1 = trava."""
    TN, FP = conf[0, 0], conf[0, 1]
    FN, TP = conf[1, 0], conf[1, 1]

    total = TN + FP + FN + TP
    accuracy = (TP + TN) / total if total > 0 else 0

    prec_trava = TP / (TP + FP) if (TP + FP) > 0 else 0
    rec_trava  = TP / (TP + FN) if (TP + FN) > 0 else 0
    iou_trava  = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0
    f1_trava   = 2 * prec_trava * rec_trava / (prec_trava + rec_trava) if (prec_trava + rec_trava) > 0 else 0

    iou_netrava = TN / (TN + FN + FP) if (TN + FN + FP) > 0 else 0
    miou = (iou_trava + iou_netrava) / 2

    return {
        'accuracy': accuracy, 'prec_trava': prec_trava, 'rec_trava': rec_trava,
        'iou_trava': iou_trava, 'f1_trava': f1_trava,
        'iou_netrava': iou_netrava, 'miou': miou,
    }

In [4]:
npy_pattern = os.path.join(DATASET_PATH, "tree_*", "*.npy")
all_npy_files = glob.glob(npy_pattern)
print(f"Oblaka: {len(all_npy_files)}")

conf = np.zeros((2, 2), dtype=np.int64)   # [stvarno][predvidjeno], 1=trava
vremena = []

for fp in tqdm(all_npy_files, desc="RANSAC po oblaku"):
    d = np.load(fp, allow_pickle=True).item()
    points = np.array(d['points'].T, dtype=np.float64)
    labels = np.array(d['labels'], dtype=np.int64)

    gt = (labels == TRAVA).astype(np.int64)   # 1 = trava, 0 = ne-trava

    t0 = time.time()
    pred = ransac_ukloni_travu(points)
    vremena.append(time.time() - t0)

    for s, p in zip(gt, pred):
        conf[s, p] += 1

m = izracunaj_metrike(conf)

print("\n" + "="*55)
print("RANSAC uklanjanje trave — rezultati na cijelom skupu")
print("="*55)
print(f"Ukupno tocaka: {conf.sum():,}")
print(f"Accuracy:          {m['accuracy']:.4f}")
print("-"*55)
print(f"IoU trava:         {m['iou_trava']:.4f}")
print(f"Preciznost trava:  {m['prec_trava']:.4f}")
print(f"Odziv trava:       {m['rec_trava']:.4f}")
print(f"F1 trava:          {m['f1_trava']:.4f}")
print("-"*55)
print(f"IoU ne-trava:      {m['iou_netrava']:.4f}")
print(f"mIoU (2 klase):    {m['miou']:.4f}")
print("-"*55)
print(f"Prosjecno vrijeme po oblaku: {np.mean(vremena):.4f} s")
print("="*55)

print("\nMatrica zabune [redak=stvarno, stupac=predvidjeno]:")
print(f"{'':<12}{'ne-trava':<12}{'trava':<12}")
print(f"{'ne-trava':<12}{conf[0,0]:<12}{conf[0,1]:<12}")
print(f"{'trava':<12}{conf[1,0]:<12}{conf[1,1]:<12}")

Oblaka: 657


RANSAC po oblaku: 100%|██████████| 657/657 [1:12:14<00:00,  6.60s/it]


RANSAC uklanjanje trave — rezultati na cijelom skupu
Ukupno tocaka: 62,163,687
Accuracy:          0.9994
-------------------------------------------------------
IoU trava:         0.9960
Preciznost trava:  0.9962
Odziv trava:       0.9998
F1 trava:          0.9980
-------------------------------------------------------
IoU ne-trava:      0.9992
mIoU (2 klase):    0.9976
-------------------------------------------------------
Prosjecno vrijeme po oblaku: 6.5332 s

Matrica zabune [redak=stvarno, stupac=predvidjeno]:
            ne-trava    trava       
ne-trava    52093837    38127       
trava       1832        10029891    


In [6]:
def boji_trava(mask):
    c = np.zeros((len(mask), 3))
    c[mask == 1] = [0.0, 0.8, 0.0]   # trava zeleno
    c[mask == 0] = [0.6, 0.6, 0.6]   # ne-trava sivo
    return c

def boji_greske(gt, pred):
    c = np.zeros((len(gt), 3))
    c[(gt==1)&(pred==1)] = [0.0, 0.8, 0.0]   # TP zeleno
    c[(gt==0)&(pred==0)] = [0.6, 0.6, 0.6]   # TN sivo
    c[(gt==0)&(pred==1)] = [1.0, 0.0, 0.0]   # FP crveno (lazno trava)
    c[(gt==1)&(pred==0)] = [0.0, 0.0, 1.0]   # FN plavo (promasena trava)
    return c

rezultati = []
for fp in all_npy_files:
    d = np.load(fp, allow_pickle=True).item()
    points = np.array(d['points'].T, dtype=np.float64)
    labels = np.array(d['labels'], dtype=np.int64)
    gt = (labels == TRAVA).astype(np.int64)
    pred = ransac_ukloni_travu(points)
    tp = np.sum((gt==1)&(pred==1)); fp_ = np.sum((gt==0)&(pred==1)); fn = np.sum((gt==1)&(pred==0))
    iou = tp/(tp+fp_+fn) if (tp+fp_+fn)>0 else (1.0 if gt.sum()==0 else 0.0)
    rezultati.append({'fp': fp, 'points': points, 'gt': gt, 'pred': pred, 'iou': iou})

rezultati.sort(key=lambda x: x['iou'])
najgori, najbolji = rezultati[0], rezultati[-1]

for tag, rez in [("NAJBOLJI", najbolji), ("NAJGORI", najgori)]:
    ime = os.path.basename(os.path.dirname(rez['fp'])) + "/" + os.path.basename(rez['fp'])
    print(f"\n[{tag}] {ime} | IoU trava: {rez['iou']:.4f}")
    pts = rez['points']

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd.colors = o3d.utility.Vector3dVector(boji_trava(rez['gt']))
    o3d.visualization.draw_geometries([pcd], window_name=f"{tag} GROUND TRUTH (zeleno=trava)")

    pcd2 = o3d.geometry.PointCloud()
    pcd2.points = o3d.utility.Vector3dVector(pts)
    pcd2.colors = o3d.utility.Vector3dVector(boji_trava(rez['pred']))
    o3d.visualization.draw_geometries([pcd2], window_name=f"{tag} RANSAC PREDIKCIJA (zeleno=trava)")

    pcd3 = o3d.geometry.PointCloud()
    pcd3.points = o3d.utility.Vector3dVector(pts)
    pcd3.colors = o3d.utility.Vector3dVector(boji_greske(rez['gt'], rez['pred']))
    o3d.visualization.draw_geometries([pcd3], window_name=f"{tag} GRESKE (zeleno=TP, sivo=TN, crveno=FP, plavo=FN)")


[NAJBOLJI] tree_1_V_0183/9.npy | IoU trava: 1.0000

[NAJGORI] tree_1_V_0005/5.npy | IoU trava: 0.0000
